In [ ]:
import thresh
# Pattern for collecting large numbers of match_dtos

with thresh.TaskStack() as ts:
    for parameters in ...:
        ts.push_task(thresh.league_exp_v4_entries(...))

    async for entry in ts.pop_results():
        ts.push_task(thresh.match_v5_matches_by_puuid(entry["id"]))

    async for match in ts.pop_results():
        ts.push_task(thresh.match_v5_matches(match["id"]))

    async for match_dto in ts.pop_results():
       # add match data to local db etc
       ...        
   
    # collect errors
    errors = ts.pop_errors()

######

for parameters in ...:
    
    entries = client.league_exp_v4_entries(parameters)

    for entry in entries:
        matches = client.match_v5_matches_by_puuid(entry["id"])

        # a lot of matches in memory!
        for match in matches:
            math_dto = client.match_v5_matches(match["id"])


In [ ]:
import thresh as trs
from thresh.concurrency import concurrently_request, Pipeline
from thresh.keywords import Unique, Limit, NodeLimit
from thresh.middlewares import retry_middleware, http_error_logging_middleware # aiohttp middlewares
from thresh.schema import puuid, match_id, Unique, MatchDto, Match


async def main():

    async with RiotAPIClient() as client:
        match[Match] = await client.match_v5_matches(...)

        matches: Iterable[Match] = await concurrently_request(client.match_v5_matches, inputs=...)

        # you can create a crawler like so. Uses concurrently request to handle large, queries
        # Unique will prevent the same match_id from being queried multiple times
        pipeline_list = [
            (client.league_exp_v4_entries, Unique(puuid), NodeLimit(100)),
            # only use unique puuids, 
            (client.match_v5_matches_by_puuid, Unique(match_id), NodeLimit(20)), # each puuid will generate 20 unique match ids       
            (client.match_v5_matches, MatchDto) # 
        ]

        middlewares_list = [
            retry_middleware(2),
            retry_middleware(20),
            retry_middleware(20)
        ]
        
        pipe = Pipeline(pipeline_list, http_error_logging_middleware)
        matches: AsyncIterator[MatchDto] = pipe(...)
